In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
#Part 1 : Text preprocessing
import re, json
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import nltk
nltk.download('stopwords')

corpus = {
 "D1": "Information retrieval helps users find relevant documents.",
 "D2": "Information Retrieval systems index documents and documents.",
 "D3": "Search engines retrieve relevant web documents.",
 "D4": "Document indexing supports efficient information retrieval.",
 "D5": "Machine learning improves search engine ranking.",
 "D6": "Search systems use indexed documents to retrieve information.",
 "D7": "The search engine retrieves documents; the documents are searchable.",
 "D8": "Indexing documents makes information retrieval more efficient.",
 "D9": "DOCUMENTS contain information about document indexing.",
 "D10": "Retrieval systems organise documents and document collections.",
}

def preprocess_document(text):
    text = text.lower()                      # minuscules
    text = re.sub(r"[^\w\s]", " ", text)     # supprime la ponctuation
    return text.split()                      # tokenisation

stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

tokens_A = {d: preprocess_document(t) for d, t in corpus.items()}
tokens_B = {d: [w for w in toks if w not in stop_words] for d, toks in tokens_A.items()}
tokens_C = {d: [stemmer.stem(w) for w in toks] for d, toks in tokens_B.items()}

for name, tk in zip("ABC", (tokens_A, tokens_B, tokens_C)):
    json.dump(tk, open(f"/content/drive/MyDrive/Colab Notebooks/results/tokens_{name}.json", "w"), indent=2)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
# Vocabulary and document representation

def build_vocabulary(tokenized_docs):
    return sorted({w for toks in tokenized_docs.values() for w in toks})

def build_term_document_matrix(tokenized_docs):
    vocab = build_vocabulary(tokenized_docs)
    return {t: {d: int(t in toks) for d, toks in tokenized_docs.items()} for t in vocab}

def build_tf_matrix(tokenized_docs):
    vocab = build_vocabulary(tokenized_docs)
    return {t: {d: toks.count(t) for d, toks in tokenized_docs.items()} for t in vocab}

import pandas as pd
for name, tk in zip("ABC", (tokens_A, tokens_B, tokens_C)):
    open(f"/content/drive/MyDrive/Colab Notebooks/results/vocabulary_{name}.txt", "w").write("\n".join(build_vocabulary(tk)))
    pd.DataFrame(build_term_document_matrix(tk)).T.to_csv(f"/content/drive/MyDrive/Colab Notebooks/results/binary_matrix_{name}.csv")
    pd.DataFrame(build_tf_matrix(tk)).T.to_csv(f"/content/drive/MyDrive/Colab Notebooks/results/tf_matrix_{name}.csv")

In [11]:
# 3.1 Index manuel pour D1 à D3

# Positions à partir de 0 :

 #Doc	 |   Tokens
 # D1	 |  information(0) retrieval(1) helps(2) users(3) find(4) relevant(5) documents(6)
 # D2	 |  information(0) retrieval(1) systems(2) index(3) documents(4) and(5) documents(6)
 # D3	 |  search(0) engines(1) retrieve(2) relevant(3) web(4) documents(5)

# Terme	       |  Index manuel
# information	 |   D1 : [0], D2 : [0]
# retrieval	   |   D1 : [1], D2 : [1]
# documents	   |   D1 : [6], D2 : [4, 6], D3 : [5]

In [12]:
# 3.2 à 3.4 Implémentation
from collections import defaultdict
import json

RESULTS = "/content/drive/MyDrive/Colab Notebooks/results"

def build_inverted_index(tokenized_documents):
    index = defaultdict(lambda: defaultdict(list))
    for doc_id, tokens in tokenized_documents.items():
        for pos, term in enumerate(tokens):
            index[term][doc_id].append(pos)
    return {t: dict(index[t]) for t in sorted(index)}

def search_term(inverted_index, term):
    return list(inverted_index.get(term, {}).keys())

inv_A = build_inverted_index(tokens_A)
inv_B = build_inverted_index(tokens_B)
inv_C = build_inverted_index(tokens_C)

for name, inv in zip("ABC", (inv_A, inv_B, inv_C)):
    json.dump(inv, open(f"{RESULTS}/inverted_index_{name}.json", "w"), indent=2)

for t in ["information", "retrieval", "documents"]:
    print(t, inv_A[t])

information {'D1': [0], 'D2': [0], 'D4': [4], 'D6': [7], 'D8': [3], 'D9': [2]}
retrieval {'D1': [1], 'D2': [1], 'D4': [5], 'D8': [4], 'D10': [0]}
documents {'D1': [6], 'D2': [4, 6], 'D3': [5], 'D6': [4], 'D7': [4, 6], 'D8': [1], 'D9': [0], 'D10': [3]}


In [13]:
# 3.5 Comparaison avec l'index manuel
manual = {
    "information": {"D1": [0], "D2": [0]},
    "retrieval":   {"D1": [1], "D2": [1]},
    "documents":   {"D1": [6], "D2": [4, 6], "D3": [5]},
}

for term, expected in manual.items():
    auto = {d: p for d, p in inv_A[term].items() if d in ("D1", "D2", "D3")}
    print(term, "OK" if auto == expected else f"ÉCART : {auto} vs {expected}")

information OK
retrieval OK
documents OK


In [14]:
# 4. Effect of preprocessing
import pandas as pd

def measures(tokenized_docs, inverted_index):
    n_docs = len(tokenized_docs)
    return {
        "vocabulary_size": len(inverted_index),
        "avg_tokens_per_doc": round(sum(len(t) for t in tokenized_docs.values()) / n_docs, 2),
        "total_postings": sum(len(docs) for docs in inverted_index.values()),
        "total_positions": sum(len(p) for docs in inverted_index.values() for p in docs.values()),
    }

rows = {
    "A (minuscules + ponctuation)": measures(tokens_A, inv_A),
    "B (A + stop-words)":           measures(tokens_B, inv_B),
    "C (B + stemming)":             measures(tokens_C, inv_C),
}
comparison = pd.DataFrame(rows).T
comparison.index.name = "version"
comparison.to_csv("/content/drive/MyDrive/Colab Notebooks/results/preprocessing_comparison.csv")
comparison

,vocabulary_size,avg_tokens_per_doc,total_postings,total_positions
version,,,,
A (minuscules + ponctuation),36.0,6.9,66.0,69.0
B (A + stop-words),30.0,6.1,59.0,61.0
C (B + stemming),24.0,6.1,57.0,61.0


In [15]:
# 5.1 et 5.2 : search_term
def docs_containing(tokenized_docs, term):
    return [d for d, toks in tokenized_docs.items() if term in toks]

versions = {"A": (tokens_A, inv_A), "B": (tokens_B, inv_B), "C": (tokens_C, inv_C)}


for term, v in [("web", "B"), ("documents", "A"), ("blockchain", "A")]:
    toks, inv = versions[v]
    res = search_term(inv, term)
    assert res == docs_containing(toks, term)
    print(f"{term!r} (version {v}) ->", res)


print("retrieval  en B ->", search_term(inv_B, "retrieval"))
print("retrieval  en C ->", search_term(inv_C, "retrieval"))
print("retrieval  en C ->", search_term(inv_C, stemmer.stem("retrieval")))

'web' (version B) -> ['D3']
'documents' (version A) -> ['D1', 'D2', 'D3', 'D6', 'D7', 'D8', 'D9', 'D10']
'blockchain' (version A) -> []
retrieval  en B -> ['D1', 'D2', 'D4', 'D8', 'D10']
retrieval  en C -> []
retrieval  en C -> ['D1', 'D2', 'D3', 'D4', 'D6', 'D7', 'D8', 'D10']


In [16]:
#5.3 : positions correctes
for name, (toks, inv) in versions.items():

    for term, postings in inv.items():
        for doc, positions in postings.items():
            for p in positions:
                assert toks[doc][p] == term, (name, term, doc, p)

    assert sum(len(p) for postings in inv.values() for p in postings.values()) \
           == sum(len(t) for t in toks.values())
    print(f"Positions version {name} : OK")

# Exemple
print(inv_A["documents"]["D2"], inv_B["documents"]["D2"])

Positions version A : OK
Positions version B : OK
Positions version C : OK
[4, 6] [4, 5]


In [17]:
# 5.4 : comparaison index automatique vs index manuel
manual = {
    "information": {"D1": [0], "D2": [0]},
    "retrieval":   {"D1": [1], "D2": [1]},
    "documents":   {"D1": [6], "D2": [4, 6], "D3": [5]},
}

ecarts = []
for term, attendu in manual.items():
    auto = {d: p for d, p in inv_A.get(term, {}).items() if d in ("D1", "D2", "D3")}
    statut = "OK" if auto == attendu else "ÉCART"
    print(f"{term:12} | manuel : {attendu} | automatique : {auto} | {statut}")
    if auto != attendu:
        ecarts.append(term)

print("\nNombre d'écarts détectés :", len(ecarts))
assert not ecarts, f"Écarts à corriger : {ecarts}"

information  | manuel : {'D1': [0], 'D2': [0]} | automatique : {'D1': [0], 'D2': [0]} | OK
retrieval    | manuel : {'D1': [1], 'D2': [1]} | automatique : {'D1': [1], 'D2': [1]} | OK
documents    | manuel : {'D1': [6], 'D2': [4, 6], 'D3': [5]} | automatique : {'D1': [6], 'D2': [4, 6], 'D3': [5]} | OK

Nombre d'écarts détectés : 0


In [18]:
#5.5 : valeurs des matrices
for name, (toks, inv) in versions.items():
    B = build_term_document_matrix(toks)
    T = build_tf_matrix(toks)
    for term in B:
        for doc in toks:
            assert B[term][doc] in (0, 1)
            assert T[term][doc] == toks[doc].count(term)
            assert (T[term][doc] > 0) == (B[term][doc] == 1)
    assert sum(T[t][d] for t in T for d in toks) == sum(len(x) for x in toks.values())
    print(f"Matrices version {name} : OK")

assert build_tf_matrix(tokens_A)["documents"]["D2"] == 2

Matrices version A : OK
Matrices version B : OK
Matrices version C : OK


***Partie 2.5 : binaire vs fréquences***

La matrice binaire enregistre uniquement la présence (1) ou l'absence (0) d'un terme dans un document. Elle ignore la répétition : un terme présent une fois et un terme présent plusieurs fois sont indiscernables. La matrice des fréquences (TF) enregistre le nombre d'occurrences de chaque terme dans chaque document (par exemple « documents » = 2 dans D2) et conserve donc l'information sur l'importance d'un terme dans un document. On peut déduire la matrice binaire de la matrice TF, mais pas l'inverse.

***Partie 3.5 : index automatique vs index manuel***

L'index généré automatiquement correspond à l'index manuel pour D1 à D3 sur les termes « information », « retrieval » et « documents » (par exemple « documents » : D1 [6], D2 [4, 6], D3 [5]). Aucun écart n'a été constaté, aucune correction n'a donc été nécessaire.


***Partie 5.4 : vérification***
aucun écart entre l'index automatique et l'index manuel.